# Leakage hunt

One row is one client in the bank's phone campaign, described as of their last call. `y` says whether they subscribed to a term deposit. The contractor's model uses every column and claims 90% accuracy. The first code cell rebuilds that model; your job starts at step 2.

## Step 1 · Run the starter

The cell loads the table, splits it 75 / 25 with `stratify=y`, and defines `fit_and_score(columns)`. That function fits the contractor's pipeline (scale the numbers, one-hot encode the text, logistic regression) on the columns you pass and prints two numbers on the test rows:

- **accuracy**: the share of test clients the model gets right;
- **hit rate**: the model ranks the test clients, the bank calls the top 10%, and the hit rate is the share of those calls that reach a subscriber.

Depending on the download route, `unknown` arrives as text or as a missing value; the starter turns missing values back into the category `"unknown"`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/bank_marketing.csv")
df = df.rename(columns={"deposit": "y", "day": "day_of_week"})    # one set of names for every download route
for col in ["job", "education", "contact", "poutcome"]:
    df[col] = df[col].fillna("unknown")

y = (df["y"] == "yes").astype(int)
X = df.drop(columns="y")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
CALLS = len(y_test) // 10                     # the bank calls the top 10% of test clients


def fit_and_score(columns, train=X_train, test=X_test):
    """Fit the contractor's pipeline on these columns; print accuracy and the call-list hit rate."""
    num = [c for c in columns if pd.api.types.is_numeric_dtype(train[c])]
    cat = [c for c in columns if c not in num]
    pre = ColumnTransformer([("num", StandardScaler(), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    model = make_pipeline(pre, LogisticRegression(max_iter=2000)).fit(train[columns], y_train)
    proba = model.predict_proba(test[columns])[:, 1]
    called = np.argsort(-proba)[:CALLS]       # the clients the model would call first
    acc = accuracy_score(y_test, model.predict(test[columns]))
    hit = y_test.values[called].mean()
    print(f"accuracy {acc:.3f} | hit rate {hit:.3f} on {CALLS} called clients")
    return model, acc, hit


print(len(X_train), "training rows,", len(X_test), "test rows")
contractor, acc_all, hit_all = fit_and_score(list(X.columns))

## Step 2 · Before or after the call?

The director asks: "Can we trust that number for next month's calls?" For each column, decide whether its value exists **before** the call you are deciding to make. The README's column table says what each one holds. Write your list in the next cell.

## Step 3 · Remove the *after* columns and refit

Call `fit_and_score` with the *before* columns only.

## Step 4 · A model that knows nothing

Score `DummyClassifier(strategy="most_frequent")` on the same split and compare its accuracy with step 1. Also print the share of subscribers in the test rows: that is the hit rate of calling clients at random.

## Step 5 · The `pdays` code

What share of rows has `pdays = -1`? A model reads `-1` as "one day less than 0". Add a column `contacted_before` (`yes` / `no`), set `pdays` to 0 where it was `-1`, and refit without the *after* columns.

## Step 6 · Two sentences for the director

What the contractor's score was, what the honest score is, and why the gap exists. Write them in a markdown cell, or print them.

## Check yourself